In [2]:
!pip install -q flask flask-cors pandas scikit-learn requests tqdm pyngrok


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import requests
import json
import time
from tqdm import tqdm
import pickle
import os
import ast
from datetime import datetime, timedelta
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading
import gc
import socket
from google.colab import output
from IPython.display import display, HTML

: 

In [4]:
TMDB_API_KEY = "3bbcbb9b9d91639fc80a860d92bb7ecc"
TMDB_BASE_URL = "https://api.themoviedb.org/3"

In [13]:
# Create data directory
!mkdir -p /content/movie_data
!mkdir -p /content/movie_data/cache

print("Setup complete!")

Setup complete!


In [14]:
class DynamicMovieFetcher:
    """Fetches Telugu movies dynamically from TMDB API"""

    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = TMDB_BASE_URL
        self.cache_dir = '/content/movie_data/cache'

    def fetch_telugu_movies_dynamic(self, max_pages=50, force_refresh=False):
        """
        Fetch Telugu movies dynamically with pagination
        Returns up-to-date movie data
        """
        all_movies = []
        seen_ids = set()

        # Try to load cached IDs to avoid duplicates
        cache_file = f"{self.cache_dir}/movie_ids_cache.json"
        if os.path.exists(cache_file) and not force_refresh:
            try:
                with open(cache_file, 'r') as f:
                    seen_ids = set(json.load(f))
                print(f"📦 Loaded {len(seen_ids)} cached movie IDs")
            except:
                pass

        print(f"🔄 Fetching Telugu movies from TMDB (max {max_pages} pages)...")

        # Fetch movies from multiple date ranges to get complete data
        date_ranges = [
            ('1900-01-01', '2020-01-01'),  # Classic movies
            ('2020-01-01', '2022-01-01'),  # Recent
            ('2022-01-01', '2023-01-01'),  # Newer
            ('2023-01-01', '2024-01-01'),  # Newest
            ('2024-01-01', datetime.now().strftime('%Y-%m-%d'))  # Latest
        ]

        for start_date, end_date in date_ranges:
            page = 1
            while page <= max_pages // len(date_ranges):
                params = {
                    'api_key': self.api_key,
                    'with_original_language': 'te',
                    'sort_by': 'popularity.desc',
                    'page': page,
                    'vote_count.gte': 10,  # Lower threshold to include more movies
                    'primary_release_date.gte': start_date,
                    'primary_release_date.lte': end_date
                }

                try:
                    response = requests.get(f"{self.base_url}/discover/movie", params=params)
                    data = response.json()

                    if 'results' not in data or not data['results']:
                        break

                    for movie in data['results']:
                        if movie['id'] not in seen_ids:
                            seen_ids.add(movie['id'])
                            all_movies.append(movie)

                    print(f"  📄 Page {page} ({start_date} to {end_date}): {len(data['results'])} movies")
                    page += 1
                    time.sleep(0.2)  # Rate limiting

                except Exception as e:
                    print(f"  ❌ Error on page {page}: {e}")
                    break

            print(f"  ✅ Found {len(all_movies)} total Telugu movies so far")

        # Save updated cache
        with open(cache_file, 'w') as f:
            json.dump(list(seen_ids), f)

        print(f"\n🎬 Total unique Telugu movies fetched: {len(all_movies)}")
        return all_movies

    def fetch_movie_details_batch(self, movies, max_movies=500):
        """
        Fetch detailed information for movies in batches
        """
        detailed_movies = []

        # Sort by popularity to get most relevant movies first
        movies_sorted = sorted(movies, key=lambda x: x.get('popularity', 0), reverse=True)
        movies_to_process = movies_sorted[:max_movies]

        print(f"\n📝 Fetching details for {len(movies_to_process)} movies...")

        for movie in tqdm(movies_to_process, desc="Processing movies"):
            details = self.fetch_movie_details(movie['id'])
            if details:
                detailed_movies.append(details)
            time.sleep(0.15)  # Rate limiting

        return detailed_movies

    def fetch_movie_details(self, movie_id):
        """Fetch detailed information for a specific movie"""
        # Check cache first
        cache_file = f"{self.cache_dir}/movie_{movie_id}.json"
        if os.path.exists(cache_file):
            try:
                with open(cache_file, 'r') as f:
                    cached_data = json.load(f)
                    # Check if cache is less than 7 days old
                    cache_age = time.time() - cached_data.get('cached_at', 0)
                    if cache_age < 7 * 24 * 3600:  # 7 days
                        return cached_data['data']
            except:
                pass

        url = f"{self.base_url}/movie/{movie_id}"
        params = {
            'api_key': self.api_key,
            'append_to_response': 'credits,keywords,similar,release_dates'
        }

        try:
            response = requests.get(url, params=params)
            data = response.json()

            # Extract genres
            genres = [genre['name'] for genre in data.get('genres', [])]

            # Extract directors and writers
            directors = []
            writers = []
            if 'credits' in data:
                for crew in data['credits'].get('crew', []):
                    if crew['job'] == 'Director':
                        directors.append(crew['name'])
                    elif crew['job'] in ['Writer', 'Screenplay', 'Story']:
                        writers.append(crew['name'])

            # Extract main actors
            actors = []
            if 'credits' in data:
                for cast in data['credits'].get('cast', [])[:8]:  # Top 8 actors
                    actors.append(cast['name'])

            # Extract keywords
            keywords = []
            if 'keywords' in data:
                keywords = [kw['name'] for kw in data['keywords'].get('keywords', [])[:15]]

            # Get release date and certification
            release_date = data.get('release_date', '')
            certification = 'NR'
            if 'release_dates' in data:
                for release in data['release_dates'].get('results', []):
                    if release['iso_3166_1'] == 'IN':
                        for rel in release.get('release_dates', []):
                            if rel.get('certification'):
                                certification = rel['certification']
                                break

            # Get similar movies from TMDB
            similar_movies = []
            if 'similar' in data and 'results' in data['similar']:
                similar_movies = [{
                    'id': sim['id'],
                    'title': sim['title'],
                    'poster_path': sim.get('poster_path', '')
                } for sim in data['similar']['results'][:5]]

            movie_data = {
                'tmdb_id': movie_id,
                'title': data.get('title', ''),
                'original_title': data.get('original_title', ''),
                'overview': data.get('overview', ''),
                'tagline': data.get('tagline', ''),
                'genres': genres,
                'actors': actors,
                'directors': directors,
                'writers': writers,
                'keywords': keywords,
                'release_date': release_date,
                'certification': certification,
                'runtime': data.get('runtime', 0),
                'vote_average': data.get('vote_average', 0),
                'vote_count': data.get('vote_count', 0),
                'popularity': data.get('popularity', 0),
                'poster_path': data.get('poster_path', ''),
                'backdrop_path': data.get('backdrop_path', ''),
                'original_language': data.get('original_language', ''),
                'status': data.get('status', ''),
                'budget': data.get('budget', 0),
                'revenue': data.get('revenue', 0),
                'similar_movies': similar_movies,
                'last_updated': datetime.now().isoformat()
            }

            # Cache the data
            cache_data = {
                'data': movie_data,
                'cached_at': time.time()
            }
            with open(cache_file, 'w') as f:
                json.dump(cache_data, f)

            return movie_data

        except Exception as e:
            print(f"Error fetching details for {movie_id}: {e}")
            return None

    def refresh_recent_movies(self, days=30):
        """
        Refresh movies released in the last X days
        """
        start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
        end_date = datetime.now().strftime('%Y-%m-%d')

        print(f"🔄 Refreshing movies from {start_date} to {end_date}...")

        recent_movies = []
        page = 1

        while page <= 10:
            params = {
                'api_key': self.api_key,
                'with_original_language': 'te',
                'sort_by': 'release_date.desc',
                'page': page,
                'primary_release_date.gte': start_date,
                'primary_release_date.lte': end_date
            }

            try:
                response = requests.get(f"{self.base_url}/discover/movie", params=params)
                data = response.json()

                if 'results' not in data or not data['results']:
                    break

                for movie in data['results']:
                    details = self.fetch_movie_details(movie['id'])
                    if details:
                        recent_movies.append(details)

                page += 1
                time.sleep(0.2)

            except Exception as e:
                print(f"Error: {e}")
                break

        return recent_movies

In [15]:
# Initialize the fetcher
fetcher = DynamicMovieFetcher(TMDB_API_KEY)

# Fetch dynamic movie data
print("="*60)
print("🎬 FETCHING LATEST TELUGU MOVIES")
print("="*60)

# Fetch movies
movies = fetcher.fetch_telugu_movies_dynamic(max_pages=40, force_refresh=False)

# Fetch detailed information for top movies
detailed_movies = fetcher.fetch_movie_details_batch(movies, max_movies=300)

# Convert to DataFrame
df = pd.DataFrame(detailed_movies)

# Add recent movies (last 30 days)
print("\n🆕 Checking for recent releases...")
recent_movies = fetcher.refresh_recent_movies(days=30)
if recent_movies:
    recent_df = pd.DataFrame(recent_movies)
    # Merge with existing, avoiding duplicates
    df = pd.concat([df, recent_df], ignore_index=True)
    df = df.drop_duplicates(subset=['tmdb_id'])

🎬 FETCHING LATEST TELUGU MOVIES
🔄 Fetching Telugu movies from TMDB (max 40 pages)...
  📄 Page 1 (1900-01-01 to 2020-01-01): 20 movies
  📄 Page 2 (1900-01-01 to 2020-01-01): 20 movies
  📄 Page 3 (1900-01-01 to 2020-01-01): 20 movies
  📄 Page 4 (1900-01-01 to 2020-01-01): 20 movies
  📄 Page 5 (1900-01-01 to 2020-01-01): 20 movies
  📄 Page 6 (1900-01-01 to 2020-01-01): 20 movies
  📄 Page 7 (1900-01-01 to 2020-01-01): 20 movies
  📄 Page 8 (1900-01-01 to 2020-01-01): 20 movies
  ✅ Found 160 total Telugu movies so far
  📄 Page 1 (2020-01-01 to 2022-01-01): 20 movies
  📄 Page 2 (2020-01-01 to 2022-01-01): 16 movies
  ✅ Found 196 total Telugu movies so far
  📄 Page 1 (2022-01-01 to 2023-01-01): 20 movies
  📄 Page 2 (2022-01-01 to 2023-01-01): 10 movies
  ✅ Found 226 total Telugu movies so far
  📄 Page 1 (2023-01-01 to 2024-01-01): 19 movies
  ✅ Found 245 total Telugu movies so far
  📄 Page 1 (2024-01-01 to 2026-04-01): 20 movies
  📄 Page 2 (2024-01-01 to 2026-04-01): 13 movies
  ✅ Found 278 to

Processing movies: 100%|██████████| 278/278 [03:19<00:00,  1.40it/s]



🆕 Checking for recent releases...
🔄 Refreshing movies from 2026-03-02 to 2026-04-01...


In [16]:
df.to_csv('/content/movie_data/telugu_movies_latest.csv', index=False)
print(f"\n✅ Final dataset: {len(df)} Telugu movies")
print(f"📅 Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Display sample
print("\n📊 Sample movies:")
print(df[['title', 'release_date', 'vote_average']].head(10))


✅ Final dataset: 290 Telugu movies
📅 Last updated: 2026-04-01 12:35:29

📊 Sample movies:
                        title release_date  vote_average
0          Bāhubali: The Epic   2025-10-29         6.000
1     Bāhubali: The Beginning   2015-07-10         7.522
2                The Rajasaab   2026-01-08         5.185
3                         RRR   2022-03-24         7.725
4  Salaar: Part 1 - Ceasefire   2023-12-21         6.684
5              Devara: Part 1   2024-09-26         6.900
6               Kalki 2898-AD   2024-06-26         6.429
7  Bāhubali 2: The Conclusion   2017-04-27         7.454
8                    Hi Nanna   2023-12-07         7.798
9                      Okkadu   2003-01-15         7.643


In [19]:

def safe_convert_to_list(x):
    """Safely convert string representation of list to actual list"""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return x.split(',') if x else []
    return []

def create_dynamic_feature_matrix(df):
    """Create TF-IDF feature vectors for content-based recommendations"""

    # Handle NaN values
    df.fillna('', inplace=True)

    # Convert string representations to lists
    for col in ['genres', 'actors', 'directors', 'writers', 'keywords']:
        if col in df.columns:
            df[col] = df[col].apply(safe_convert_to_list)
            df[col] = df[col].apply(lambda x: x if isinstance(x, list) else [])

    # Combine features with weighted importance
    def combine_features(row):
        # Weight different features differently
        title_weight = 3
        overview_weight = 2
        genres_weight = 2
        actors_weight = 1.5
        directors_weight = 2
        keywords_weight = 1.5

        title_text = str(row['title']) * title_weight
        overview_text = str(row['overview']) * overview_weight

        genres_text = (' '.join(row['genres']) +' ') * int(genres_weight) if isinstance(row['genres'], list) else ''
        actors_text = (' '.join(row['actors']) + ' ') * int(actors_weight) if isinstance(row['actors'], list) else ''
        directors_text = (' '.join(row['directors']) + ' ') * int(directors_weight) if isinstance(row['directors'], list) else ''
        keywords_text = (' '.join(row['keywords']) +' ') * int(keywords_weight) if isinstance(row['keywords'], list) else ''

        return ' '.join([
            title_text.lower(),
            overview_text.lower(),
            genres_text.lower(),
            actors_text.lower(),
            directors_text.lower(),
            keywords_text.lower()
        ])

    df['combined_features'] = df.apply(combine_features, axis=1)

    # Create TF-IDF vectors
    tfidf = TfidfVectorizer(
        stop_words='english',
        max_features=8000,
        min_df=2,
        ngram_range=(1, 3)  # Include trigrams for better matching
    )

    tfidf_matrix = tfidf.fit_transform(df['combined_features'])

    return tfidf_matrix, tfidf, df

# Create feature matrix
print("\n🔧 Creating feature vectors...")
tfidf_matrix, tfidf, df = create_dynamic_feature_matrix(df)

# Calculate similarity matrix
print("📊 Calculating similarity matrix...")
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"✅ TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"✅ Cosine similarity matrix shape: {cosine_sim.shape}")


🔧 Creating feature vectors...
📊 Calculating similarity matrix...
✅ TF-IDF matrix shape: (290, 3086)
✅ Cosine similarity matrix shape: (290, 290)


In [20]:
# Save models
print("💾 Saving models...")
pickle.dump(tfidf, open('/content/movie_data/tfidf_dynamic.pkl', 'wb'))
pickle.dump(cosine_sim, open('/content/movie_data/cosine_sim_dynamic.pkl', 'wb'))

💾 Saving models...


In [21]:
# Create mappings
indices = pd.Series(df.index, index=df['tmdb_id']).to_dict()
movie_titles = pd.Series(df['title'].values, index=df['tmdb_id']).to_dict()

with open('/content/movie_data/indices_dynamic.pkl', 'wb') as f:
    pickle.dump(indices, f)
with open('/content/movie_data/movie_titles_dynamic.pkl', 'wb') as f:
    pickle.dump(movie_titles, f)

print("✅ Models saved successfully!")

✅ Models saved successfully!


In [22]:

"""## Step 3: Enhanced Recommendation Functions"""

def get_recommendations_by_movie(movie_id, cosine_sim=cosine_sim, indices=indices, df=df, n=10):
    """Get movie recommendations based on a movie ID"""
    if movie_id not in indices:
        return []

    idx = indices[movie_id]

    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get top n movies (excluding itself)
    sim_scores = sim_scores[1:n+1]
    movie_indices = [i[0] for i in sim_scores]

    recommendations = df.iloc[movie_indices].to_dict('records')

    # Add similarity scores and ensure proper data types
    for i, rec in enumerate(recommendations):
        rec['similarity_score'] = float(sim_scores[i][1])
        rec['tmdb_id'] = int(rec['tmdb_id'])

        # Clean up for JSON
        for col in ['genres', 'actors', 'directors', 'writers', 'keywords']:
            if col in rec and not isinstance(rec[col], list):
                rec[col] = safe_convert_to_list(rec[col])

    return recommendations

def get_recommendations_by_search(query, df=df, tfidf=tfidf):
    """Get movie recommendations based on search query"""
    if not query or len(query.strip()) == 0:
        return []

    # Transform query to TF-IDF vector
    query_vector = tfidf.transform([query.lower()])

    # Calculate similarity with all movies
    similarities = cosine_similarity(query_vector, tfidf.transform(df['combined_features']))

    # Get top recommendations
    sim_scores = list(enumerate(similarities[0]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get top 20
    sim_scores = sim_scores[:20]
    movie_indices = [i[0] for i in sim_scores]

    recommendations = df.iloc[movie_indices].to_dict('records')

    # Add similarity scores
    for i, rec in enumerate(recommendations):
        rec['similarity_score'] = float(sim_scores[i][1])
        rec['tmdb_id'] = int(rec['tmdb_id'])

        for col in ['genres', 'actors', 'directors', 'writers', 'keywords']:
            if col in rec and not isinstance(rec[col], list):
                rec[col] = safe_convert_to_list(rec[col])

    return recommendations

def search_by_actor(actor_name, df=df):
    """Search movies by actor name"""
    matching_movies = []
    actor_lower = actor_name.lower()

    for idx, row in df.iterrows():
        actors = row.get('actors', [])
        if not isinstance(actors, list):
            actors = safe_convert_to_list(actors)

        if any(actor_lower in actor.lower() for actor in actors):
            movie_dict = row.to_dict()
            movie_dict['tmdb_id'] = int(movie_dict['tmdb_id'])

            for col in ['genres', 'actors', 'directors', 'writers', 'keywords']:
                if col in movie_dict and not isinstance(movie_dict[col], list):
                    movie_dict[col] = safe_convert_to_list(movie_dict[col])

            matching_movies.append(movie_dict)

    return matching_movies[:20]

def search_by_director(director_name, df=df):
    """Search movies by director name"""
    matching_movies = []
    director_lower = director_name.lower()

    for idx, row in df.iterrows():
        directors = row.get('directors', [])
        if not isinstance(directors, list):
            directors = safe_convert_to_list(directors)

        if any(director_lower in director.lower() for director in directors):
            movie_dict = row.to_dict()
            movie_dict['tmdb_id'] = int(movie_dict['tmdb_id'])

            for col in ['genres', 'actors', 'directors', 'writers', 'keywords']:
                if col in movie_dict and not isinstance(movie_dict[col], list):
                    movie_dict[col] = safe_convert_to_list(movie_dict[col])

            matching_movies.append(movie_dict)

    return matching_movies[:20]

def get_trending_movies(limit=10):
    """Get currently trending Telugu movies"""
    try:
        response = requests.get(
            f"{TMDB_BASE_URL}/trending/movie/week",
            params={'api_key': TMDB_API_KEY}
        )
        data = response.json()

        trending = []
        for movie in data.get('results', []):
            if movie.get('original_language') == 'te':
                movie_data = fetcher.fetch_movie_details(movie['id'])
                if movie_data:
                    trending.append(movie_data)

        return trending[:limit]
    except Exception as e:
        print(f"Error fetching trending: {e}")
        return []

def get_new_releases(limit=10):
    """Get latest Telugu movie releases"""
    return get_recommendations_by_search("new release", limit)

# Test functions
print("\n🧪 Testing recommendation functions...")

if len(df) > 0:
    test_movie = df.iloc[0]
    print(f"\n📽️ Testing recommendations for: {test_movie['title']}")
    recs = get_recommendations_by_movie(test_movie['tmdb_id'], n=5)
    for rec in recs:
        print(f"  → {rec['title']} (similarity: {rec['similarity_score']:.3f})")

print("\n🔍 Testing search for 'action thriller':")
results = get_recommendations_by_search("action thriller")
for result in results[:5]:
    print(f"  → {result['title']} (score: {result['similarity_score']:.3f})")


🧪 Testing recommendation functions...

📽️ Testing recommendations for: Bāhubali: The Epic
  → Bāhubali 2: The Conclusion (similarity: 0.354)
  → Bāhubali: The Beginning (similarity: 0.242)
  → Vikramarkudu (similarity: 0.121)
  → Kannappa (similarity: 0.111)
  → Karthikeya 2 (similarity: 0.111)

🔍 Testing search for 'action thriller':
  → Oosaravelli (score: 0.249)
  → Billa (score: 0.226)
  → 1: Nenokkadine (score: 0.226)
  → Eagle (score: 0.224)
  → Spy (score: 0.217)


In [23]:

"""## Step 4: Flask API Server with Real-time Updates"""

# Create Flask app
app = Flask(__name__)
CORS(app)

# Global variables
user_sessions = {}
last_refresh_time = time.time()
REFRESH_INTERVAL = 3600  # Refresh every hour

def refresh_movie_data():
    """Background thread to refresh movie data periodically"""
    global df, tfidf, cosine_sim, indices, last_refresh_time

    while True:
        try:
            current_time = time.time()
            if current_time - last_refresh_time > REFRESH_INTERVAL:
                print("\n🔄 Refreshing movie data...")

                # Fetch new releases
                new_movies = fetcher.refresh_recent_movies(days=7)

                if new_movies:
                    new_df = pd.DataFrame(new_movies)
                    # Merge with existing
                    df = pd.concat([df, new_df], ignore_index=True)
                    df = df.drop_duplicates(subset=['tmdb_id'])

                    # Rebuild feature matrix
                    tfidf_matrix, tfidf, df = create_dynamic_feature_matrix(df)
                    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

                    # Update indices
                    indices = pd.Series(df.index, index=df['tmdb_id']).to_dict()

                    # Save updated models
                    pickle.dump(tfidf, open('/content/movie_data/tfidf_dynamic.pkl', 'wb'))
                    pickle.dump(cosine_sim, open('/content/movie_data/cosine_sim_dynamic.pkl', 'wb'))

                    print(f"✅ Refreshed: {len(new_movies)} new movies added")

                last_refresh_time = current_time

        except Exception as e:
            print(f"❌ Refresh error: {e}")

        time.sleep(REFRESH_INTERVAL)
# Start background refresh thread
refresh_thread = threading.Thread(target=refresh_movie_data, daemon=True)
refresh_thread.start()


In [24]:
@app.route('/api/recommendations/movie/<int:movie_id>', methods=['GET'])
def get_movie_recommendations(movie_id):
    """Get recommendations based on a specific movie"""
    try:
        n = int(request.args.get('limit', 10))
        recommendations = get_recommendations_by_movie(movie_id, n=n)
        return jsonify({
            'ok': True,
            'recommendations': recommendations
        })
    except Exception as e:
        return jsonify({'ok': False, 'error': str(e)}), 500

@app.route('/api/recommendations/search', methods=['GET'])
def search_recommendations():
    """Get recommendations based on search query"""
    try:
        query = request.args.get('q', '')
        if not query:
            return jsonify({'ok': False, 'error': 'No query provided'}), 400

        # Try actor search first
        actor_results = search_by_actor(query)
        if len(actor_results) > 0:
            return jsonify({
                'ok': True,
                'type': 'actor',
                'recommendations': actor_results[:20]
            })

        # Try director search
        director_results = search_by_director(query)
        if len(director_results) > 0:
            return jsonify({
                'ok': True,
                'type': 'director',
                'recommendations': director_results[:20]
            })

        # Default to content-based search
        recommendations = get_recommendations_by_search(query)

        return jsonify({
            'ok': True,
            'type': 'search',
            'recommendations': recommendations[:20]
        })

    except Exception as e:
        return jsonify({'ok': False, 'error': str(e)}), 500

@app.route('/api/movies/tmdb/<int:tmdb_id>/recommendations', methods=['GET'])
def get_movie_recommendations_by_tmdb(tmdb_id):
    """Get recommendations by TMDB ID"""
    try:
        n = int(request.args.get('limit', 8))
        recommendations = get_recommendations_by_movie(tmdb_id, n=n)
        return jsonify({
            'ok': True,
            'recommendations': recommendations
        })
    except Exception as e:
        return jsonify({'ok': False, 'error': str(e)}), 500

@app.route('/api/session/click', methods=['POST'])
def track_click():
    """Track movie clicks for session-based recommendations"""
    try:
        data = request.get_json()
        session_id = data.get('session_id', 'default')
        movie_id = data.get('movie_id')
        movie_data = data.get('movie_data', {})

        if session_id not in user_sessions:
            user_sessions[session_id] = {
                'clicked_movies': [],
                'recommendations': []
            }

        # Add to clicked movies
        user_sessions[session_id]['clicked_movies'].append({
            'movie_id': movie_id,
            'movie_data': movie_data,
            'timestamp': time.time()
        })

        # Keep only last 10 clicks
        user_sessions[session_id]['clicked_movies'] = \
            user_sessions[session_id]['clicked_movies'][-10:]

        # Generate new recommendations based on all clicks
        all_recommendations = []
        seen_ids = set()

        for click in reversed(user_sessions[session_id]['clicked_movies'][-5:]):
            if click['movie_id'] in indices:
                recs = get_recommendations_by_movie(click['movie_id'], n=3)
                for rec in recs:
                    if rec['tmdb_id'] not in seen_ids:
                        seen_ids.add(rec['tmdb_id'])
                        all_recommendations.append({
                            **rec,
                            'reason': f"Because you liked {click['movie_data'].get('title', 'this movie')}"
                        })

        user_sessions[session_id]['recommendations'] = all_recommendations[:20]

        return jsonify({
            'ok': True,
            'recommendations': user_sessions[session_id]['recommendations']
        })

    except Exception as e:
        return jsonify({'ok': False, 'error': str(e)}), 500

@app.route('/api/session/recommendations', methods=['GET'])
def get_session_recommendations():
    """Get session-based recommendations"""
    try:
        session_id = request.args.get('session_id', 'default')
        if session_id in user_sessions:
            return jsonify({
                'ok': True,
                'recommendations': user_sessions[session_id]['recommendations']
            })
        return jsonify({'ok': True, 'recommendations': []})
    except Exception as e:
        return jsonify({'ok': False, 'error': str(e)}), 500

@app.route('/api/movies/trending', methods=['GET'])
def get_trending():
    """Get trending Telugu movies"""
    try:
        limit = int(request.args.get('limit', 10))
        trending = get_trending_movies(limit)
        return jsonify({
            'ok': True,
            'movies': trending
        })
    except Exception as e:
        return jsonify({'ok': False, 'error': str(e)}), 500

@app.route('/api/movies/new', methods=['GET'])
def get_new():
    """Get new releases"""
    try:
        limit = int(request.args.get('limit', 10))
        new_movies = get_new_releases(limit)
        return jsonify({
            'ok': True,
            'movies': new_movies
        })
    except Exception as e:
        return jsonify({'ok': False, 'error': str(e)}), 500

@app.route('/api/movies/all', methods=['GET'])
def get_all_movies():
    """Get all movies with pagination"""
    try:
        page = int(request.args.get('page', 1))
        limit = int(request.args.get('limit', 50))

        start = (page - 1) * limit
        end = start + limit

        movies = df.iloc[start:end].to_dict('records')

        # Clean for JSON
        for movie in movies:
            movie['tmdb_id'] = int(movie['tmdb_id'])
            for col in ['genres', 'actors', 'directors', 'writers', 'keywords']:
                if col in movie and not isinstance(movie[col], list):
                    movie[col] = safe_convert_to_list(movie[col])

        return jsonify({
            'ok': True,
            'movies': movies,
            'total': len(df),
            'page': page,
            'limit': limit
        })
    except Exception as e:
        return jsonify({'ok': False, 'error': str(e)}), 500

@app.route('/api/health', methods=['GET'])
def health_check():
    """Health check endpoint"""
    return jsonify({
        'ok': True,
        'status': 'running',
        'movies_count': len(df),
        'last_updated': df['last_updated'].max() if 'last_updated' in df.columns else 'N/A',
        'api_status': 'active'
    })

print("\n✅ Flask API ready!")


✅ Flask API ready!


In [28]:
"""## Step 5: Run the Server (No ngrok)"""

def get_local_ip():
    """Get local IP address"""
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
        return ip
    except:
        return "localhost"

# Get local IP
local_ip = get_local_ip()

print("\n" + "="*70)
print("🚀 TELUGU MOVIE RECOMMENDATION API SERVER")
print("="*70)
print(f"📍 Local URL: http://{local_ip}:5000")
print(f"📍 Local URL: http://localhost:5000")
print(f"🔍 Health Check: http://{local_ip}:5000/api/health")
print(f"🎬 Movies Count: {len(df)}")
print(f"🔄 Auto-refresh: Every {REFRESH_INTERVAL//3600} hour(s)")
print("="*70)
print("\n📌 API ENDPOINTS:")
print(f"  • Movie Recommendations: http://{local_ip}:5000/api/recommendations/movie/<movie_id>")
print(f"  • Search: http://{local_ip}:5000/api/recommendations/search?q=<query>")
print(f"  • Trending: http://{local_ip}:5000/api/movies/trending")
print(f"  • New Releases: http://{local_ip}:5000/api/movies/new")
print(f"  • All Movies: http://{local_ip}:5000/api/movies/all")
print(f"  • Session Clicks: POST http://{local_ip}:5000/api/session/click")
print(f"  • Session Recommendations: http://{local_ip}:5000/api/session/recommendations")
print(f"  • Server Info: http://{local_ip}:5000/api/info")
print("="*70)
print("\n💡 HOW TO CONNECT YOUR REACT APP:")
print(f"  Create .env file in your React project:")
print(f"  VITE_API_BASE=http://{local_ip}:5000")
print("="*70)
print("\n⚠️  IMPORTANT NOTES:")
print("  1. This server runs on localhost - accessible only from this machine")
print("  2. To expose to internet, use: ngrok, localtunnel, or deploy to cloud")
print("  3. Keep this notebook running to keep the API active")
print("  4. The server auto-refreshes movie data every hour")
print("="*70)
print("\n✅ Server is starting... Press Ctrl+C to stop\n")

# Run the Flask app
if __name__ == '__main__':
    # Run with debug=False and use_reloader=False to avoid double execution
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

# This code will keep running until you stop the cell
print("\n✅ Server is running in the background!")


🚀 TELUGU MOVIE RECOMMENDATION API SERVER
📍 Local URL: http://172.28.0.12:5000
📍 Local URL: http://localhost:5000
🔍 Health Check: http://172.28.0.12:5000/api/health
🎬 Movies Count: 290
🔄 Auto-refresh: Every 1 hour(s)

📌 API ENDPOINTS:
  • Movie Recommendations: http://172.28.0.12:5000/api/recommendations/movie/<movie_id>
  • Search: http://172.28.0.12:5000/api/recommendations/search?q=<query>
  • Trending: http://172.28.0.12:5000/api/movies/trending
  • New Releases: http://172.28.0.12:5000/api/movies/new
  • All Movies: http://172.28.0.12:5000/api/movies/all
  • Session Clicks: POST http://172.28.0.12:5000/api/session/click
  • Session Recommendations: http://172.28.0.12:5000/api/session/recommendations
  • Server Info: http://172.28.0.12:5000/api/info

💡 HOW TO CONNECT YOUR REACT APP:
  Create .env file in your React project:
  VITE_API_BASE=http://172.28.0.12:5000

⚠️  IMPORTANT NOTES:
  1. This server runs on localhost - accessible only from this machine
  2. To expose to internet,

INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



✅ Server is running in the background!
